# Data Formatting

The scripts here convert datasets into the formats used for the dataanalysis

## Events Manual Labels

Convert events encodings from CSV to NC

In [1]:
import numpy as np
import pandas as pd
import xarray as xr

events_coding_files = ["waddendrifters2023_grounding_events_manual_coding","waddendrifters2023_grounding_events_manual_coding_validation"]

for ec_file in events_coding_files:
  df = pd.read_csv("data/in/%s.csv"%ec_file, engine="c",header=5)
  indices = np.array([[int(n) for n in ts_name.split('_')] for ts_name in df['ts_name']])
  df['irecord'] = indices[:,0]
  df['its'] = indices[:,1]
  df = df.drop(columns=['ts_name', 'comment'])
  ds = xr.Dataset.from_dataframe(df)
  sections = np.unique([[int(ir),int(its)] for ir in ds.irecord for its in ds.its],axis=0)
  ds.attrs = {"occurring_irecord" : sections[:,0], "occurring_its" : sections[:,1]}
  ds.to_netcdf('data/out/%s.nc'%ec_file)
  print("written data/out/%s.nc"%ec_file)

ds

written data/out/waddendrifters2023_grounding_events_manual_coding.nc
written data/out/waddendrifters2023_grounding_events_manual_coding_validation.nc


<xarray.Dataset> Size: 4kB
Dimensions:     (index: 92)
Coordinates:
  * index       (index) int64 736B 0 1 2 3 4 5 6 7 8 ... 84 85 86 87 88 89 90 91
Data variables:
    ind_start   (index) int64 736B 186 190 1157 1210 ... 1184 1213 1226 1227
    tag         (index) object 736B 'O' 'O' 'D' 'W' 'D' ... 'W' 'D' 'W' 'D' 'B'
    edit notes  (index) object 736B nan nan nan nan nan ... nan nan nan nan nan
    irecord     (index) int64 736B 1 1 1 1 1 1 1 1 1 ... 24 24 26 26 26 26 26 26
    its         (index) int64 736B 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
Attributes:
    occurring_irecord:  [ 1  5  9 13 18 20 21 23 24 26]
    occurring_its:      [0 0 0 0 0 0 0 0 0 0]

# Waterlevels from MATROOS SWAN produce

In [ ]:
print("INFO: no need to run this this section with the provided input data")
print(">> waterlevel_NAP_from_Nov14_to_Dec05_2023_MATROOS_maps2d_swan_kuststrook_harmonie.npy")

In [ ]:
output_json_file = "waterlevel_NAP_from_Nov14_to_Dec05_2023_MATROOS_maps2d_swan_kuststrook_harmonie"

import netCDF4
import numpy as np
import json
import xarray as xr
import pandas as pd
import numpy as np

dat = xr.open_dataset("maps2d_swan_kuststrook_harmonie_202311140000.nc", engine="netcdf4")
dat
MMDDs = ["1114", "1115", "1116", "1117", "1118", "1119", "1120", "1121", "1122", "1123", "1124", "1125", "1126", "1127", "1128", "1129", "1130", "1201", "1202", "1203", "1204"]

waterlevel = {'time' : [], 'set' : []}
for mmdd in MMDDs:
  print(mmdd)
  data = netCDF4.Dataset('maps2d_swan_kuststrook_harmonie_2023'+mmdd+'0000.nc', format='NETCDF4')

  nt = len(data['time'][:])
  nr = len(data['row'][:])
  nc = len(data['col'][:])

  data_t = data['time'][:]*60
  data_wl = data['sep'][:].filled(fill_value=np.nan)
  data_lon = data['lon'][:].filled(fill_value=np.nan)
  data_lat = data['lat'][:].filled(fill_value=np.nan)

  # count number of valid positions
  npositions = 0
  for ir in range(nr):
    for ic in range(nc):
      if np.isnan(data_lon[ir,ic]) or np.isnan(data_lat[ir,ic]):
        continue
      npositions += 1

  for it in range(nt):
    waterlevel['time'] += [data_t[it]]
    dat_lon = np.zeros(npositions)
    dat_lat = np.zeros(npositions)
    dat_level = np.zeros(npositions)
    ind = 0
    for ir in range(nr):
      for ic in range(nc):
        if np.isnan(data_lon[ir,ic]) or np.isnan(data_lat[ir,ic]):
          continue
        dat_level[ind] = data_wl[it,ir,ic]
        dat_lon[ind] = data_lon[ir,ic]
        dat_lat[ind] = data_lat[ir,ic]
        ind += 1
    data_set = {'lon' : dat_lon, 'lat' : dat_lat, 'level' : dat_level}
    waterlevel['set'] += [data_set]

np.save(output_json_file, waterlevel)